# circuit-tracer native edge test on Gemma-2-2B (flagship)

`circuit-tracer` builds an attribution graph on a transcoder **replacement** model and
reports which feature (and error) nodes influence a target logit. A graph edge is a
candidate, not a verdict. `saturn_pub.interop.circuit_tracer.native_edge_test` re-tests each
edge on the **real** Gemma-2-2B weights and asks two honest questions, under a frozen rule:

1. **native-necessity** — is removing this one feature individually load-bearing on the real
   model? A single feature that is not is *expected* under a redundant circuit and is **not**
   by itself evidence against the graph.
2. **graph_vs_native** — does the native drop agree with the drop circuit-tracer's *own*
   `feature_intervention` predicts on the replacement model? `invert` = graph predicts a
   large effect the real model does not show; `agree_small` = both small (redundancy);
   `agree_large` = both load-bearing.

The prompt panel, admission rule, thresholds, `top_k`, group interventions, prediction modes
and statistics were **preregistered** (`experiments/circuit_tracer/PREREGISTRATION.md`,
`panel.json`) before the run. This notebook loads the committed, scrubbed `summary.json` and
bundles and re-derives every label **in-process, offline** — no GPU, no model, no
`circuit_tracer`, no torch. Install with `pip install -e .`.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "experiments").exists() else ROOT.parent
EXP = ROOT / "experiments" / "circuit_tracer"
BUNDLES = EXP / "bundles"
summary = json.loads((EXP / "summary.json").read_text())
adm = summary["admission"]
sf = summary["single_feature"]["overall"]


def ci(d):
    return "n/a" if not d else f"{d['point']:.3f} [{d['lo']:.3f}, {d['hi']:.3f}] (n={d['n']})"


print("model             :", summary["model"])
print("panel sha256      :", summary["panel_sha256"])
print("top_k             :", summary["top_k"])
print("jobs              :", ", ".join(summary["jobs"]))
print(
    "candidates/admitted:",
    adm["n_candidates"],
    "/",
    adm["n_admitted"],
    " dropped",
    adm["n_dropped"],
)
print()
print("single-feature edges tested :", sf["n_edges"])
print(
    "  native_necessary / absent / inconclusive:",
    sf["n_native_necessary"],
    "/",
    sf["n_native_effect_absent"],
    "/",
    sf["n_inconclusive"],
)
print("  agree rate  (95% CI):", ci(sf["agree_rate"]))
print("  invert rate (95% CI):", ci(sf["invert_rate"]))
print("  mixed rate  (95% CI):", ci(sf["mixed_rate"]))
es = summary["error_node_influence_share"]
print(
    "error-node influence share  : min %.3f  median %.3f  max %.3f  (n=%d)"
    % (es["min"], es["median"], es["max"], es["n"])
    if es
    else "error share: n/a"
)

## Admission and per-family agreement

A candidate is admitted iff the native greedy top-1 equals the target; failures are recorded
and dropped (never replaced). Single-feature graph agreement is broken out per task family
with a percentile bootstrap 95% CI.

In [ ]:
print(
    f"{'family':>22} {'cand':>5} {'adm':>4} {'edges':>6} {'agree 95% CI':>26} {'inv':>4} {'mix':>4}"
)
bf = summary["single_feature"]["by_family"]
for fam, a in adm["per_family"].items():
    row = bf.get(fam)
    if row:
        print(
            f"{fam:>22} {a['candidates']:>5} {a['admitted']:>4} {row['n_edges']:>6} "
            f"{ci(row['agree_rate']):>26} {row['invert']:>4} {row['mixed']:>4}"
        )
    else:
        print(f"{fam:>22} {a['candidates']:>5} {a['admitted']:>4} {'-':>6} {'(none admitted)':>26}")
if adm["dropped"]:
    print()
    print("dropped (recorded, not replaced):")
    for name, d in adm["dropped"].items():
        print(f"  {name:>14}  reason={d['reason']}  native_top1_id={d.get('native_top1_id')}")

## Group interventions and the Dallas-type gap

The paper's group interventions — jointly zero-ablate the whole top-k set, and steer it to
−2× activation — are graded natively and compared to circuit-tracer's joint prediction. The
graph prediction is computed under three `feature_intervention` modes: **frozen-attention**
(the circuit-tracer default, used for the `graph_vs_native` tag), **unconstrained** (attention
reacts), and **direct-effects** (the fully linearized attribution). If the graph overpredicts
under the default but matches native once attention is unfrozen, the gap is a property of the
prediction mode, not the graph.

In [ ]:
g = summary["group_interventions"]
for gname, v in g["agreement"].items():
    print(f"=== {gname} (n={v['n_prompts']}) ===")
    print("  agree rate (default/frozen-attn mode, 95% CI):", ci(v["agree_rate_default_mode"]))
    print(
        f"  agree_large {v['n_agree_large']}  agree_small {v['n_agree_small']}  "
        f"invert {v['n_invert']}  mixed {v['n_mixed']}  |  flips native top-1: {v['n_flip_native_top1']}"
    )
    d = g["diagnosis_by_mode"][gname]
    print("  mean |graph_pred - native| by mode:")
    for mode in ("frozen_attention_iterative", "unconstrained_iterative", "direct_effects"):
        m = d[mode]
        print(
            f"    {mode:>26}: mean {m['mean_abs_pred_minus_native']}  "
            f"median {m['median_abs_pred_minus_native']}  overpredict(>0.4) {m['n_overpredict_gt_0p4']}/{m['n']}"
        )
    if d["invert_prompts"]:
        print("  invert prompts (native vs per-mode graph prediction):")
        for p in d["invert_prompts"]:
            print(
                f"    {p['name']:>14}: native {p['native']:.3f} | frozen {p['pred_frozen']} "
                f"| unconstrained {p['pred_unconstrained']} | direct {p['pred_direct_effects']}"
            )
    print()

## Per-prompt tables (from the sealed bundles)

One compact line per admitted prompt, then the full edge table for the two-hop example.

In [ ]:
def load_table(name):
    return json.loads((BUNDLES / name / "edge_verdicts.json").read_text())


names = sorted(p.name for p in BUNDLES.iterdir() if (p / "edge_verdicts.json").exists())
print(f"{'prompt':>14} {'target':>14} {'nec/abs/inc':>12} {'agree/inv/mix':>14} {'err_share':>9}")
for name in names:
    t = load_table(name)
    s = t["summary"]
    nai = "%d/%d/%d" % (s["n_native_necessary"], s["n_native_effect_absent"], s["n_inconclusive"])
    aim = "%d/%d/%d" % (s["n_agree_with_graph"], s["n_invert_vs_graph"], s["n_mixed_vs_graph"])
    print(
        f"{name:>14} {t['target_text']!r:>14} {nai:>12} {aim:>14} "
        f"{s['error_node_influence_share']:>9.3f}"
    )

In [ ]:
def fmt(x, w=9, p=4):
    return (f"%{w}.{p}f" % x) if isinstance(x, (int, float)) else f"{str(x):>{w}}"


def show(name):
    t = load_table(name)
    print(f"=== {name} ===  prompt {t['prompt']!r} -> target {t['target_text']!r}")
    print(
        f"  {'layer':>5} {'pos':>4} {'feat':>6} {'native_d':>9} {'graph_d':>9} "
        f"{'graph_vs_native':>16} {'native_label':>20}"
    )
    for r in t["rows"]:
        if r["node_kind"] != "feature":
            continue
        gp = r.get("graph_predicted_delta")
        print(
            f"  {r['layer']!s:>5} {r['position']!s:>4} {r['feature']!s:>6} "
            f"{fmt(r['native_logit_delta'])} {fmt(gp) if gp is not None else ' NA':>9} "
            f"{str(r.get('graph_vs_native')):>16} {r['verdict']:>20}"
        )
    for gg in t.get("group_interventions", []):
        bm = gg.get("graph_predicted_delta_by_mode", {})
        bmr = {k: (round(v, 4) if v is not None else None) for k, v in bm.items()}
        print(
            f"  GROUP {gg['name']:>16} (m={gg['multiplier']:>4}): native {fmt(gg['native_logit_delta'])}"
            f" -> {gg['graph_vs_native']} / {gg['verdict']} (flip {gg['top_token_flip']})"
        )
        print("        graph pred by mode:", bmr)


example = "mh_dallas" if "mh_dallas" in names else names[0]
show(example)

## Fresh-process re-derivation (offline, stdlib)

`verify_edge_bundle` re-checks each bundle's SHA-256 and re-derives every covered edge label,
its `graph_vs_native` tag, and every group verdict from the sealed scalars under the frozen
`DecisionRule` — no torch, no model, no `circuit_tracer`.

In [ ]:
from saturn_pub.interop.circuit_tracer import verify_edge_bundle

allok = True
for name in names:
    r = verify_edge_bundle(BUNDLES / name)
    allok = allok and r["ok"]
    if not r["ok"]:
        print(
            f"{name}: ok={r['ok']} hash={r['hash_ok']} fp={r['fingerprint_ok']} "
            f"verdict={r['verdict_ok']} group={r['group_ok']}"
        )
print("all", len(names), "bundles re-derive offline:", allok)
assert allok

## Reading

`native_effect_absent` on a single feature is only the first axis: under a redundant circuit
many features carry little alone, so it is expected and not a graph failure. The
**`graph_vs_native`** column is the fair test, and the **group** interventions are where the
paper expects a real effect. The three-mode diagnosis separates a true *graph* overprediction
from a *prediction-mode* artifact. See [`docs/circuit-tracer.md`](../docs/circuit-tracer.md)
and [`experiments/circuit_tracer/README.md`](../experiments/circuit_tracer/README.md).